In [1]:
%matplotlib tk

In [ ]:
import dnois
from dnois.optics import rt
import torch
from polysurf import ExtendedPolynomial


c:\Users\18520\anaconda3\envs\dnois_py311\Lib\tkinter\__init__.py:861: UserWarning: Glyph 8711 (\N{NABLA}) missing from font(s) Microsoft YaHei.
  func(*args)
c:\Users\18520\anaconda3\envs\dnois_py311\Lib\tkinter\__init__.py:861: UserWarning: Glyph 8711 (\N{NABLA}) missing from font(s) Microsoft YaHei.
  func(*args)
c:\Users\18520\anaconda3\envs\dnois_py311\Lib\tkinter\__init__.py:861: UserWarning: Glyph 8711 (\N{NABLA}) missing from font(s) Microsoft YaHei.
  func(*args)


In [3]:
torch.set_default_dtype(torch.double)
torch.set_grad_enabled(False)

In [4]:
# 对比修改前后 ExtendedPolynomial 面型的差异
from compare_polysurf import compare_surfaces

# 使用第3个面（索引2）的参数进行对比
roc = -613.98
conic = 18.807
b = [0., 0., -2.35, 0., -2.664, 0., 0.126, 0., 0.096, 0.148, 0., 0.2840, 0.139]
aperture = rt.RectangularAperture(21 * 2, 20 * 2, center_y=29)
norm_radius = 50

# 执行对比分析
results = compare_surfaces(
    roc=roc,
    conic=conic,
    b=b,
    aperture=aperture,
    norm_radius=norm_radius,
    grid_size=100,
    save_path='polysurf_comparison.png'
)


d:\Github\dnois\demos\202511离轴\compare_polysurf.py:264: UserWarning: Glyph 8711 (\N{NABLA}) missing from font(s) Microsoft YaHei.
  plt.savefig(save_path, dpi=150, bbox_inches='tight')


对比图已保存到: polysurf_comparison.png

详细统计信息
矢高差异统计 (mm):
  最大值: 0.00e+00
  最小值: 0.00e+00
  平均值: 0.00e+00
  标准差: 0.00e+00
  均方根: 0.00e+00
  最大绝对值: 0.00e+00

梯度差异统计:
  最大模: 5.72e-17
  平均模: 4.71e-18
  均方根模: 1.01e-17


In [5]:
dnois.set_default('length', 'mm')
dnois.set_default('angle', 'deg')
dnois.ext.zmx.load_agf('CHINA.agf')

In [6]:
def apply_tilt_x(context, angle):
    context.phi = 90
    context.theta = -angle
    context.chi = -90

In [8]:
sq = rt.SurfaceSequence([
    rt.Plane(aperture=31.675),
    rt.Plane(aperture=24.424),
    ExtendedPolynomial(-613.98, 18.807, [
        0., 0., -2.35, 0., -2.664, 0., 0.126, 0., 0.096, 0.148, 0., 0.2840, 0., 0.139,
    ], aperture=rt.RectangularAperture(21 * 2, 20 * 2, center_y=29), norm_radius=50, reflective=True),
    rt.Conic(-71.19, 1.645, aperture=8.2, reflective=True),
    ExtendedPolynomial(-60.58, -0.523, [
        0., 0., 6.668, 0., 6.906, 0., 0.03, 0., 0.032, 0.354, 0., 0.762, 0., 0.424,
    ], aperture=rt.RectangularAperture(22.5 * 2, 21.5 * 2, center_y=-19.9), norm_radius=50, reflective=True),
    rt.Plane(aperture=10.004),
    rt.Plane('K9', rt.RectangularAperture(8.95 * 2, 8.5 * 2)),
    rt.Plane(aperture=rt.RectangularAperture(8.95 * 2, 8.5 * 2)),
    rt.Plane(aperture=rt.RectangularAperture(4.4 * 2, 3.7 * 2)),  #IMAGE
])
sq.all_relative_()
sq[1].context.z = 80.
sq[2].context.z = 55.
sq[2].context.y = -27.67
apply_tilt_x(sq[2].context, -11.74)
sq[3].context.z = -55.
sq[3].context.y = 5.12
sq[4].context.z = 55
sq[4].context.y = 2.03 - 5.12
sq[5].context.z = -55
sq[5].context.y = -2.03 - 16.147
apply_tilt_x(sq[5].context, 1.92)
sq[6].context.z = -18.048
sq[7].context.z = -0.7
sq[8].context.z = -0.4

for idx in [3, 5, 6, 7, 8]:
    sq[idx].context.upward_in = False

In [9]:
optics = rt.CoaxialRayTracing(sq)

In [10]:
optics.plot_3d((0., 0.))

In [11]:
# 由于 SurfaceSequence 没有 trace_out() 方法，需要手动实现点列图计算

import matplotlib.pyplot as plt
import math
from dnois.optics.rt.crt.vis import CRTSpotDiagram
from dnois.optics.rt import BatchedRay
from dnois.optics.rt.crt.core import _make_direction
from dnois import utils, base

def plot_spot_diagram_offaxis(
    optics,
    points=None,
    wl=None,
    ray_density=6,
    width=None,
    entr_d=None,
    entr_z=None,
):
    """
    为离轴系统计算点列图
    适配 SurfaceSequence
    """
    if wl is None:
        wl = optics.wl
    if isinstance(wl, (float, int)):
        wl = torch.tensor([wl], device=optics.device, dtype=optics.dtype)
    wl = wl.to(optics.device)
    
    if points is None:
        # 默认使用几个视场点
        fov_half = optics.reference.fov_half if hasattr(optics, 'reference') else 1.0
        fov = [0., fov_half * 0.5 ** 0.5, fov_half]
        points = optics.fovd2obj([(0., fov_item) for fov_item in fov], float('inf'))
    
    points = optics.cam2lens(points)
    n_point = points.size(0)
    n_row = int(math.sqrt(n_point) + 1e-5)
    n_col = int(math.ceil(n_point / n_row))
    fig, axs = plt.subplots(n_row, n_col, squeeze=False, figsize=(n_col * 5, n_row * 5))
    
    # 确定入瞳参数
    if entr_d is None or entr_z is None:
        try:
            entr_r_computed, entr_z_computed = optics.entr_pupil('paraxial', wl, 'center')
            if entr_d is None:
                entr_d = entr_r_computed.item() * 2
            if entr_z is None:
                entr_z = entr_z_computed.item()
        except:
            # 如果无法计算入瞳，使用第一个表面的孔径
            if entr_d is None:
                entr_d = 2 * optics.surfaces.first.aperture.max_radius().item()
            if entr_z is None:
                entr_z = 0.0
    
    # 在入瞳处采样
    from dnois.optics.rt.surf import CircularAperture
    pupil_ap = CircularAperture(entr_d / 2)
    pupil_ap.to(device=optics.device, dtype=optics.dtype)
    x, y = pupil_ap.sample_unipolar(ray_density, 6)
    pupil_points = torch.stack([x, y, torch.full_like(x, entr_z)], -1)  # N_spp x 3
    entr_center = optics.new_tensor([0, 0, entr_z])
    
    # 获取图像平面（最后一个表面）的位置和方向
    img_surface = optics.surfaces.last
    img_plane_origin = img_surface.context.abs_origin  # 图像平面的全局位置
    img_plane_normal = img_surface.context.abs_rm @ torch.tensor([0., 0., 1.], device=optics.device, dtype=optics.dtype)
    
    rms_list = []
    geo_radius_list = []
    
    for i in range(n_point):
        # 追迹主光线束
        direction, _ = _make_direction(pupil_points, points[i])  # N_spp x 3
        ray_in = BatchedRay(pupil_points, direction, wl.view(-1, 1))  # N_wl x N_spp
        
        # 使用 trace() 追迹到最后一个表面（图像平面）
        try:
            ray_out = optics.surfaces.trace(ray_in)  # N_wl x N_spp
        except Exception as e:
            print(f"警告: 视场点 {i+1} 的光线束追迹失败: {e}")
            # 创建一个空的结果，但保持正确的形状
            n_spp = pupil_points.size(0)
            ray_out = BatchedRay(
                torch.zeros((wl.size(0), n_spp, 3), device=optics.device, dtype=optics.dtype),
                torch.zeros((wl.size(0), n_spp, 3), device=optics.device, dtype=optics.dtype),
                wl.view(-1, 1).expand(-1, n_spp)
            )
            # 将所有光线标记为无效
            ray_out.valid = torch.zeros((wl.size(0), n_spp), dtype=torch.bool, device=optics.device)
        
        # 追迹主光线（chief ray）
        # 如果主光线无法追迹，使用有效光线的中心作为主光线
        chief_direction, _ = _make_direction(entr_center, points[i])  # 3
        chief_ray_in = BatchedRay(entr_center, chief_direction, wl.unsqueeze(-1))  # N_wl x 1
        
        try:
            chief_ray_out = optics.surfaces.trace(chief_ray_in, aperture=False)  # N_wl x 1，不使用光阑
        except:
            # 如果主光线无法追迹，使用有效光线的中心作为主光线
            if ray_out.valid.any():
                # 计算有效光线的中心位置
                valid_mask = ray_out.valid  # N_wl x N_spp
                valid_positions = ray_out.o  # N_wl x N_spp x 3
                # 对每个波长，计算有效光线的平均位置
                chief_positions = []
                for wl_idx in range(wl.size(0)):
                    if valid_mask[wl_idx].any():
                        mean_pos = valid_positions[wl_idx][valid_mask[wl_idx]].mean(dim=0)  # 3
                    else:
                        mean_pos = torch.zeros(3, device=optics.device, dtype=optics.dtype)
                    chief_positions.append(mean_pos)
                chief_positions = torch.stack(chief_positions).unsqueeze(1)  # N_wl x 1 x 3
                # 创建一个虚拟的主光线
                chief_ray_out = BatchedRay(
                    chief_positions,
                    torch.zeros_like(chief_positions),
                    wl.unsqueeze(-1)
                )
            else:
                print(f"警告: 视场点 {i+1} 没有有效光线，跳过")
                # 创建一个空的主光线
                chief_ray_out = BatchedRay(
                    torch.zeros((wl.size(0), 1, 3), device=optics.device, dtype=optics.dtype),
                    torch.zeros((wl.size(0), 1, 3), device=optics.device, dtype=optics.dtype),
                    wl.unsqueeze(-1)
                )
        
        # 将光线坐标转换到图像平面的局部坐标系
        # 图像平面的局部坐标系：z轴垂直于图像平面
        ray_out_local = img_surface.context.g2l(ray_out.o, direction=False)  # N_wl x N_spp x 3
        chief_ray_out_local = img_surface.context.g2l(chief_ray_out.o, direction=False)  # N_wl x 1 x 3
        
        # 计算相对于主光线的偏移（在图像平面上的 x, y 坐标）
        # 确保形状正确：ray_out_local 是 N_wl x N_spp x 3，chief_ray_out_local 是 N_wl x 1 x 3
        # chief_ray_out_local[..., 0] 是 N_wl x 1，会自动广播到 N_wl x N_spp
        x = ray_out_local[..., 0] - chief_ray_out_local[..., 0]  # N_wl x N_spp
        y = ray_out_local[..., 1] - chief_ray_out_local[..., 1]  # N_wl x N_spp
        
        # 检查是否有有效光线
        if not ray_out.valid.any():
            print(f"警告: 视场点 {i+1} 没有有效光线，跳过")
            # 为每个波长添加 NaN 值
            for wl_idx in range(wl.size(0)):
                rms_list.append(torch.tensor(float('nan'), device=optics.device, dtype=optics.dtype))
                geo_radius_list.append(torch.tensor(float('nan'), device=optics.device, dtype=optics.dtype))
            # 绘制空图
            r, c = i // n_col, i % n_col
            ax: plt.Axes = axs[r][c]
            ax.text(0.5, 0.5, 'No valid rays', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'Field point {i+1} (No valid rays)')
            continue
        
        # 计算 RMS 和几何半径
        # ray_out.valid 的形状是 N_wl x N_spp，需要确保 r2 也是这个形状
        r2 = x.square() + y.square()  # N_wl x N_spp
        
        # 对每个波长分别计算
        for wl_idx in range(wl.size(0)):
            valid_mask = ray_out.valid[wl_idx]  # N_spp
            if valid_mask.any():
                r2_valid = r2[wl_idx][valid_mask]  # N_valid
                rms_list.append(r2_valid.mean().sqrt())
                geo_radius_list.append(r2_valid.max().sqrt())
            else:
                rms_list.append(torch.tensor(float('nan'), device=optics.device, dtype=optics.dtype))
                geo_radius_list.append(torch.tensor(float('nan'), device=optics.device, dtype=optics.dtype))
        
        # 绘制点列图（使用第一个波长的数据）
        r, c = i // n_col, i % n_col
        ax: plt.Axes = axs[r][c]
        for j in range(wl.size(0)):
            wl_value = wl[j].item()
            valid_mask = ray_out.valid[j]  # N_spp
            if valid_mask.any():
                ax.scatter(
                    utils.t4plot(x[j][valid_mask]), 
                    utils.t4plot(y[j][valid_mask]),
                    s=2, 
                    c=utils.wl2rgb(wl_value, output_format='hex'), 
                    label=base.Length.fmt(wl_value, 'um'),
                )
        ax.legend()
        ax.set_aspect('equal')
        if width is not None:
            ax.set_xlim(-width / 2, width / 2)
            ax.set_ylim(-width / 2, width / 2)
        ax.set_xlabel('x (mm)')
        ax.set_ylabel('y (mm)')
        ax.grid(True, alpha=0.3)
        ax.set_title(f'Field point {i+1}')
    
    # 重新组织 RMS 和几何半径：每个视场点对应每个波长
    # 当前 rms_list 和 geo_radius_list 的长度是 n_point * n_wl
    # 需要重新组织为 n_point 个张量，每个张量包含 n_wl 个值
    n_wl = wl.size(0)
    rms_reshaped = [torch.stack(rms_list[i*n_wl:(i+1)*n_wl]) for i in range(n_point)]
    geo_radius_reshaped = [torch.stack(geo_radius_list[i*n_wl:(i+1)*n_wl]) for i in range(n_point)]
    
    rms = torch.stack(rms_reshaped)  # n_point x n_wl
    geo_radius = torch.stack(geo_radius_reshaped)  # n_point x n_wl
    return CRTSpotDiagram(fig, rms, geo_radius)

print("点列图计算函数已定义")


点列图计算函数已定义


In [15]:
# ============================================================
# 计算点列图
# ============================================================

# 设置参数
wl = torch.tensor([0.7e-3])  
ray_density = 15  # 光线密度
width = 0.01  # 点列图显示范围 (mm)

# 定义视场点（物空间方向）
# fov_angles = [(0., 0.)]  # 视场角（度）
fov_angles = [(0., 0.), (4, 0.), (0., 3.3)]  # 视场角（度）
obj_points = optics.fovd2obj(fov_angles, float('inf'), in_degrees=True)  # 转换为物空间坐标

print(f"视场点数量: {obj_points.shape[0]}")
print(f"波长: {wl[0].item()*1e6:.1f} nm")
print(f"光线密度: {ray_density}")
print(f"显示范围: ±{width/2:.3f} mm")

# 计算点列图
spot_diagram = plot_spot_diagram_offaxis(
    optics,
    points=obj_points,
    wl=wl,
    ray_density=ray_density,
    width=width,
)

# 显示结果
print(f"\n点列图统计:")
print(f"RMS (均方根半径): {spot_diagram.rms * 1000} μm")
print(f"几何半径: {spot_diagram.geo_radius * 1000} μm")

plt.tight_layout()
plt.show()


视场点数量: 3
波长: 700.0 nm
光线密度: 15
显示范围: ±0.005 mm

点列图统计:
RMS (均方根半径): tensor([[1.5308],
        [3.7722],
        [2.4146]]) μm
几何半径: tensor([[4.3288],
        [6.2925],
        [3.7700]]) μm
